# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sheby-me/FLYRANK-Onboarding/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: One row represents one pseudonymized content item for a specific client on a specific day (grain: report_date × client_id × content_id).
Time Window: A mid-panel month, given March 2026 (month=2026-03), to avoid developing inside the final test window.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

fact_table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

print("--- Grain Check ---")
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM read_parquet('{fact_table}')
    GROUP BY 1, 2, 3
    HAVING row_count > 1
    LIMIT 5
""").df()
print(grain_check)

--- Grain Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features: gsc_avg_position, ctr, scroll_rate, ai_traffic_pct, ga4_data_available (These are knowable at the decision moment before predicting).

Features: gsc_impressions, gsc_avg_position, ga4_sessions, ga4_data_available (Raw daily metrics knowable at the decision moment).

Label / Proxy: gsc_clicks (We will use the volume of clicks as our proxy target to predict).

Context: client_hash_id, content_hash_id, report_date (Used strictly for grouping, joining, and train/test splits; never as features).

Excluded: Using the current day's gsc_clicks as a feature to predict the current day's gsc_clicks. This is direct target leakage and would give the model a perfect, but completely useless, score.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Here we verify the row count, date span, and data availability for our slice. We also build a feature frame that explicitly omits the label-derived leak columns (trend_pct) to keep the prediction honest.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Row count and Date span check
print("--- Date Window & Row Count ---")
span_check = con.sql(f"""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date,
        COUNT(*) as total_rows
    FROM read_parquet('{fact_table}')
""").df()
print(span_check)

# 2. Availability check (Filtering with IS TRUE)
print("\n--- GA4 Data Availability ---")
availability_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as usable_rows,
        ROUND((COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*)), 2) as pct_usable
    FROM read_parquet('{fact_table}')
""").df()
print(availability_check)

# 3. The Feature Frame & The Trap
print("\n--- Feature Frame (Trap Removed) ---")
feature_frame = con.sql(f"""
    SELECT
        -- Context (UPDATED TO HASH IDs)
        client_hash_id, content_hash_id, report_date,

        -- Proxy Label (What we want to predict)
        gsc_clicks AS target_clicks,

        -- Features (Safe, daily metrics)
        gsc_impressions,
        gsc_avg_position,
        ga4_sessions,
        ga4_data_available

        -- THE TRAP: Direct Target Leakage
        -- If we leave current 'gsc_clicks' in as a feature while trying to predict 'target_clicks',
        -- the model will just map it 1:1 and learn nothing.
        -- gsc_clicks <-- COMMENTED OUT to prevent leakage.
    FROM read_parquet('{fact_table}')
    WHERE ga4_data_available IS TRUE
    LIMIT 5
""").df()
print(feature_frame)

--- Date Window & Row Count ---
  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378

--- GA4 Data Availability ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  usable_rows  pct_usable
0     9841378       413966        4.21

--- Feature Frame (Trap Removed) ---
            client_hash_id           content_hash_id report_date  \
0  client_65de48885f4ef01b  content_09be8cc7fcb222af  2026-03-01   
1  client_65de48885f4ef01b  content_851afac9fe13612e  2026-03-01   
2  client_65de48885f4ef01b  content_cee6c6fc8c51af14  2026-03-01   
3  client_65de48885f4ef01b  content_5e120e972f11f833  2026-03-01   
4  client_65de48885f4ef01b  content_16a7291bb6ecaebe  2026-03-01   

   target_clicks  gsc_impressions  gsc_avg_position  ga4_sessions  \
0              0                0               NaN             1   
1              0                0               NaN             1   
2              0                0               NaN             1   
3              0                0               NaN             1   
4              0                0               NaN             1   

   ga4_data_available  
0                True  
1           

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
What this data can never tell you:

Unbalanced Client History: We cannot assume all clients have data starting at the same time. The history depth differs wildly per client, so we must check dim_clients.gsc_data_start and ga4_data_start rather than assuming a global calendar window.

Missingness vs. Zero Engagement: Rows appearing before a client's ga4_data_start will have GA4 columns zero-filled with ga4_data_available = FALSE. These zeroes represent missing analytics access, not a lack of user engagement.

Missing Keyword Data: Missingness often follows the content_type (e.g., some types have ~100% missing keyword data). A blind fillna(0) would accidentally inject a category signal.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No query strictly required here, but we can print an acknowledgement.
print("Limits acknowledged: Missing GA4 data is not zero engagement, and history depth is heavily unbalanced across clients.")

Limits acknowledged: Missing GA4 data is not zero engagement, and history depth is heavily unbalanced across clients.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.